<a href="https://colab.research.google.com/github/biolographer/NanoparticlesSAM/blob/main/sam_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SEM particle annotation


In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')
nb_path = '/content/notebooks'
models_path = '../checkpoints/'
os.symlink('/content/drive/My Drive/Colab Notebooks/packages', nb_path)
os.symlink('/content/drive/My Drive/Colab Notebooks/models', models_path)
sys.path.insert(0,nb_path)

Mounted at /content/drive


In [ ]:
using_colab = True

In [ ]:
if using_colab:
    import torch
    import torchvision
    print("PyTorch version:", torch.__version__)
    print("Torchvision version:", torchvision.__version__)
    print("CUDA is available:", torch.cuda.is_available())
    import sys
    #!{sys.executable} -m pip install opencv-python matplotlib
    !{sys.executable} -m pip install --target=$nb_path 'git+https://github.com/facebookresearch/sam2.git'

    !mkdir -p $models_path
    !wget -nc -P $models_path https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt


In [2]:
%load_ext autoreload
%autoreload 2

import os
import sys

if not os.path.isdir('./NanoparticlesSAM'):
    !git clone https://github.com/Biolographer/NanoparticlesSAM.git

module_path = os.path.abspath(os.path.join('NanoparticlesSAM/NanoparticlesSAM'))

#module_path = os.path.abspath(os.path.join('./NanoparticlesSAM'))
sys.path.append(module_path)

Cloning into 'NanoparticlesSAM'...
remote: Enumerating objects: 277, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 277 (delta 53), reused 47 (delta 21), pack-reused 182 (from 2)
Receiving objects: 100% (277/277), 123.84 MiB | 15.22 MiB/s, done.
Resolving deltas: 100% (119/119), done.


In [3]:
from NanoparticlesSAM import *
from particle_seg import *
from particle_loader import *
from plots import plot_seg_mask, plot_rect


In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import glob
import torchvision
from tqdm import tqdm

import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

from PIL import Image

#%matplotlib tk
import matplotlib.image as mpimg

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())


PyTorch version: 2.5.1+cu124
Torchvision version: 0.20.1+cu124
CUDA is available: True


# configure model

In [ ]:
# select the device for computation
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":
    # use bfloat16 for the entire notebook
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
elif device.type == "mps":
    print(
        "\nSupport for MPS devices is preliminary. SAM 2 is trained with CUDA and might "
        "give numerically different outputs and sometimes degraded performance on MPS. "
        "See e.g. https://github.com/pytorch/pytorch/issues/84936 for a discussion."
    )

In [ ]:
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

sam2_checkpoint = "../checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"

sam2 = build_sam2(model_cfg, sam2_checkpoint, device=device, apply_postprocessing=False)

mask_generator = SAM2AutomaticMaskGenerator(sam2)

# perform segmentation and save results

In [10]:
folder_path = '/content/drive/MyDrive/shield_data/20250228'
subfolders = [f.path for f in os.scandir(folder_path) if f.is_dir()]
print(subfolders)

['/content/drive/MyDrive/shield_data/20250228/A3-1h', '/content/drive/MyDrive/shield_data/20250228/A4-2h', '/content/drive/MyDrive/shield_data/20250228/A2-40min', '/content/drive/MyDrive/shield_data/20250228/A1-20min-01', '/content/drive/MyDrive/shield_data/20250228/Immo']


In [ ]:
for subfolder in tqdm(subfolders, desc="Processing folders", unit="folder"):

  img_set = Particle_Dataset(subfolder, device='jeol')
  save_path = os.path.join(os.getcwd(),'results', img_set.root.split('/')[-1])

  if not os.path.isdir(save_path):
    os.makedirs(save_path, exist_ok=True)

  subfolder_df_list = []

  for idx in tqdm(range(len(img_set)), desc="Processing images", unit="img"):

      #Get image
      img, name, metadata = img_set.__getitem__(idx)
      name = img_set.files[idx]

      if metadata:
        nm_per_pixel = float(metadata['CM_PIXEL_SIZE'].split('nm')[0])
      else:
        nm_per_pixel = None

      saving_name = name.split('.')[0]
      print(f'Analyzing particle: {saving_name}')

      #Apply the SAM method
      combined_mask, simple_mask, dataframe_SAM = sphere_segmentation(img, mask_generator,
                                                                      nanometer_per_pixel=nm_per_pixel,
                                                                      min_diameter_cutoff=150,
                                                                      max_diameter_cutoff=300,
                                                                      circularity_cutoff = 0.65,
                                                                      border_cutoff=True,
                                                                      max_feret_filter=False,
                                                                      min_feret_filter=False,
                                                                      hough_circles=False)

      dataframe_SAM['img_name'] = saving_name
      dataframe_SAM['experiment'] = save_path.split('/')[-1]
      dataframe_SAM['nm_per_pixel'] = nm_per_pixel

      #add results to list
      if not dataframe_SAM.empty and isinstance(combined_mask, np.ndarray):
        subfolder_df_list.append(dataframe_SAM)

        #plot results
        plot_seg_mask(img, combined_mask, save=True,
                      name=f'{saving_name}', save_path=save_path)
        print(f'\nimage from {saving_name} saved!\n')
      else:
        with open(f'{save_path}/failed_analysis.txt', 'a') as f:
          f.write(f'{saving_name} could not be analyzed')


  if subfolder_df_list != []:
    #save results
    save_df = pd.concat(subfolder_df_list)
    save_df.drop('segmentation', axis=1, inplace=True)
    experiment = save_path.split('/')[-1]
    save_df.to_pickle(f'{save_path}/df_analyzed_{experiment}.pkl')

In [ ]:
import shutil

source_folder = '/content/results'
destination_folder = os.path.join(folder_path, '..', 'results')

if not os.path.exists(destination_folder):
  os.makedirs(destination_folder)

shutil.copytree(source_folder, destination_folder, dirs_exist_ok=True)

print(f"Folder '{source_folder}' copied to '{destination_folder}'")

Folder '/content/results' copied to '/content/drive/MyDrive/shield_data/20250228/..'


In [ ]:
!zip -qr /content/results.zip /content/results

from google.colab import files

files.download("/content/results.zip")
